In [1]:
import pickle
from pathlib import Path
import pandas as pd
from scipy import stats

Opens WESAD folder and opens every .pkl file to collect wrist data for each patient before concatenating into a singular dataset. Final dataframe has a column for each participant's signal and emotion label at recording. Labels were downsampled from 700hz to 4hz by taking the mode of each divided section of the signal.

In [ ]:
# Set directory with WESAD data here
wesad_dir = Path("../Data/wesad_eda/WESAD")
# Empty list to store all the participants
subject_list = []
# Loop to open all the files
for i in wesad_dir.rglob("*.pkl"):
    subject_id = i.parent.name
    with open(i, "rb") as file:
        data = pickle.load(file, encoding='latin1')
        wrist_data = data['signal']['wrist']
        eda_signal = wrist_data['EDA'].flatten()
        #subject_id = data['label']
        # APPARENTLY the label is for 700hz and this needs to be recounted to match labels to 4hz accordingly
        label_700hz = data['label'].flatten()
        # 700/4=175 so we take chunks every 175 then label based on mode rule, I am open to suggestions for downsampling the labels
        n_chunks = len(label_700hz) // 175
        label_chunks = label_700hz[: n_chunks * 175].reshape(n_chunks, 175)
        label_4hz = stats.mode(label_chunks, axis=1).mode.flatten()
        # Rounding issue with getting mismatched vector lengths for concatenation
        min_len = min(len(eda_signal), len(label_4hz))
        # Build Dataframe
        df_subject = pd.DataFrame({
            'EDA': eda_signal[:min_len],
            'Label': label_4hz[:min_len]
            })
        df_subject.columns = [subject_id + '_EDA', subject_id + '_Label']
        #df_subject['Subject'] = subject_id
        subject_list.append(df_subject)
#Make the final dataframe        
eda_final = pd.concat(subject_list, axis=1)

Save final dataset to a csv file.

In [10]:
with open("wesad_subject_list.pkl", "wb") as f:
    pickle.dump(subject_list, f)

```
# Relabel numbers for easy human readable labels
final_df.loc[final_df['Label'] == 0, 'Label'] = 'Transient'
final_df.loc[final_df['Label'] == 1, 'Label'] = 'Baseline'
final_df.loc[final_df['Label'] == 2, 'Label'] = 'Stress'
final_df.loc[final_df['Label'] == 3, 'Label'] = 'Amusement'
final_df.loc[final_df['Label'] == 4, 'Label'] = 'Meditation'
final_df.loc[final_df['Label'] == 5, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 6, 'Label'] = 'Ignore'
final_df.loc[final_df['Label'] == 7, 'Label'] = 'Ignore'
final_df['Label'].unique()
```

We see each subject's EDA has different length.

In [6]:
for i in range(len(subject_list)):
    print(f"{len(subject_list[i])}")

21008
20952
25972
24316
22192
21864
20932
23680
21984
22524
25692
22148
25032
20892
28284
